# Notebook 04 - Data Cleaning with Pandas

**Topic:** Data Processing with Pandas  
**Subtopic:** Cleaning

---

## What you will learn

- How to detect and handle missing values using multiple strategies
- How to find and remove duplicate records
- How to fix inconsistent data types and string formatting
- How to detect and treat outliers
- How to validate data against business rules
- How to build a reusable cleaning pipeline

---

## Prerequisites

```bash
pip install pandas numpy matplotlib seaborn
```

---

## Section 1 - Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

np.random.seed(42)
os.makedirs("output", exist_ok=True)

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Ready.")

---

## Section 2 - Creating a Dirty Dataset

We deliberately inject the most common real-world data quality problems into a synthetic employee dataset. This gives us full control over what problems exist and lets us verify each fix.

**Problems we will inject:**
- Missing values across multiple columns
- Duplicate rows
- Inconsistent string casing and whitespace
- Mixed data types in a numeric column
- Outliers (salary of 1,000,000)
- Invalid values (negative age, future join date)
- Inconsistent category labels (same value spelled differently)

In [ ]:
raw_data = {
    "employee_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
                    111, 112, 113, 114, 115, 102, 108, 116, 117, 118],
    "name": ["Alice Martin", "bob smith", "Carol White", "  David Lee ", "Eve Johnson",
             "Frank Brown", "Grace Kim", "HENRY DAVIS", "Iris Wilson", "Jack Moore",
             "Karen Hall", "Leo Young", "Mia Allen", "Noah Scott", "Olivia King",
             "bob smith", "HENRY DAVIS", None, "Quinn Adams", "Ryan Baker"],
    "department": ["Engineering", "Marketing", "engineering", "HR", "Marketing",
                   "Finance", "Engineering", "HR", "Finance", "MARKETING",
                   "Engineering", "hr", "Finance", "Engineering", "Marketing",
                   "Marketing", "HR", "Finance", None, "Engineering"],
    "age": [29, 34, 41, -5, 28, 52, 36, 45, 31, 27,
            38, 44, 29, 33, 26, 34, 45, 39, 30, None],
    "salary": [72000, 58000, 95000, 64000, "N/A", 88000, 102000, 77000, 91000, 53000,
               1000000, 68000, 85000, 110000, 61000, 58000, 77000, 73000, None, 67000],
    "join_date": ["2019-03-15", "2021-07-22", "2018-11-01", "2020-05-10", "2023-01-30",
                  "2015-09-14", "2017-04-28", "2022-08-05", "2019-12-19", "2024-02-11",
                  "2016-06-30", "2020-10-07", "2021-03-22", "2018-07-15", "2023-11-09",
                  "2021-07-22", "2022-08-05", "2030-01-01", "2020-04-18", "2019-09-25"],
    "performance_score": [4.2, 3.8, None, 4.5, 3.1, 4.0, 4.7, 3.5, None, 3.9,
                          4.1, 3.6, 4.3, 4.8, 3.2, 3.8, 3.5, 4.0, 3.7, None],
    "email": ["alice@company.com", "bob@company.com", "carol@company.com", "david@company.com",
              "eve@company.com", "frank@company.com", "grace@company.com", "henry@company.com",
              "iris@company.com", "jack@company.com", "karen@company.com", "leo@company.com",
              "mia@company.com", "noah@company.com", "olivia@company.com", "bob@company.com",
              "henry@company.com", "peter@company.com", "quinn@company.com", "ryan@company.com"],
}

df_dirty = pd.DataFrame(raw_data)
print(f"Shape: {df_dirty.shape}")
df_dirty

---

## Section 3 - Step 1: Initial Inspection

Before touching anything, always inspect the dataset completely. Never assume you know what is wrong.

In [ ]:
print("Data types:")
print(df_dirty.dtypes)
print()
print("Missing values:")
missing = df_dirty.isnull().sum()
missing_pct = (missing / len(df_dirty) * 100).round(1)
print(pd.DataFrame({"count": missing, "pct": missing_pct}).to_string())

In [ ]:
print("Unique values per column:")
for col in df_dirty.columns:
    print(f"  {col:<22} {df_dirty[col].nunique()} unique")

In [ ]:
# Notice salary is 'object' dtype, not numeric - a key sign of mixed types
print("Salary column sample values:")
print(df_dirty["salary"].tolist())

---

## Section 4 - Step 2: Fix Data Types

Data type problems must be fixed first because many subsequent operations (e.g. calculating means, comparing dates) will silently produce wrong results or raise errors if types are wrong.

In [ ]:
df = df_dirty.copy()  # Always work on a copy

# Convert salary: coerce turns non-numeric values ("N/A") into NaN
df["salary"] = pd.to_numeric(df["salary"], errors="coerce")

# Convert join_date to proper datetime
df["join_date"] = pd.to_datetime(df["join_date"], errors="coerce")

print("Data types after conversion:")
print(df.dtypes)
print()
print(f"Salary NaN count after coerce: {df['salary'].isnull().sum()}")

---

## Section 5 - Step 3: Fix String Inconsistencies

Inconsistent casing, leading/trailing whitespace, and variant spellings are among the most common data quality issues in string columns. They cause group-by operations to split records that should be together.

In [ ]:
# Before cleaning
print("Department values BEFORE cleaning:")
print(df["department"].value_counts().to_string())

In [ ]:
# Fix names: strip whitespace, apply title case
df["name"] = df["name"].str.strip().str.title()

# Fix department: strip and title case
df["department"] = df["department"].str.strip().str.title()

# Fix email: lowercase (emails are case-insensitive)
df["email"] = df["email"].str.lower().str.strip()

print("Department values AFTER cleaning:")
print(df["department"].value_counts().to_string())

Notice how `engineering`, `Engineering`, and `MARKETING` all collapse into the correct category after `.str.title()`.

---

## Section 6 - Step 4: Remove Duplicates

Duplicate rows inflate counts and distort averages. We identify them by `employee_id` because that is the natural unique key, not by matching all columns.

In [ ]:
print(f"Rows before deduplication: {len(df)}")

# Show which rows are duplicates
dupes = df[df.duplicated(subset="employee_id", keep=False)].sort_values("employee_id")
print(f"\nDuplicate employee_id rows ({len(dupes)} rows):")
print(dupes[["employee_id", "name", "department", "salary"]].to_string())

In [ ]:
# Keep the first occurrence of each employee_id
df = df.drop_duplicates(subset="employee_id", keep="first").reset_index(drop=True)

print(f"Rows after deduplication: {len(df)}")

---

## Section 7 - Step 5: Validate Against Business Rules

Business rule validation catches values that are syntactically valid but logically impossible. No data type conversion will catch a negative age or a future join date.

In [ ]:
today = pd.Timestamp("today").normalize()

# Rule 1: Age must be between 18 and 80
invalid_age = df[(df["age"] < 18) | (df["age"] > 80)]
print(f"Invalid age rows: {len(invalid_age)}")
print(invalid_age[["employee_id", "name", "age"]].to_string())

# Rule 2: join_date must not be in the future
future_dates = df[df["join_date"] > today]
print(f"\nFuture join_date rows: {len(future_dates)}")
print(future_dates[["employee_id", "name", "join_date"]].to_string())

# Rule 3: performance_score must be between 0 and 5
invalid_score = df[(df["performance_score"] < 0) | (df["performance_score"] > 5)]
print(f"\nInvalid performance_score rows: {len(invalid_score)}")

In [ ]:
# Set invalid values to NaN so they are handled in the imputation step
df.loc[(df["age"] < 18) | (df["age"] > 80), "age"] = np.nan
df.loc[df["join_date"] > today, "join_date"] = pd.NaT

print("Invalid values replaced with NaN.")
print(f"Age nulls now   : {df['age'].isnull().sum()}")
print(f"Date nulls now  : {df['join_date'].isnull().sum()}")

---

## Section 8 - Step 6: Detect and Handle Outliers

Outliers are extreme values that distort statistics. The two most common detection methods are the IQR method (distribution-based) and Z-score (standard deviation-based).

**IQR method:** Any value below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR` is flagged as an outlier.

In [ ]:
def detect_outliers_iqr(series):
    """
    Return a boolean mask where True indicates an outlier,
    using the interquartile range method.
    """
    q1  = series.quantile(0.25)
    q3  = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)


salary_outliers = detect_outliers_iqr(df["salary"].dropna())
outlier_idx = salary_outliers[salary_outliers].index

print("Salary outliers detected:")
print(df.loc[outlier_idx, ["employee_id", "name", "salary"]].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2)

df["salary"].plot(kind="box", ax=axes[0], vert=False)
axes[0].set_title("Salary distribution (with outlier)")
axes[0].set_xlabel("Salary (USD)")

# Cap the outlier at the 95th percentile (winsorisation)
cap_value = df["salary"].quantile(0.95)
df["salary"] = df["salary"].clip(upper=cap_value)

df["salary"].plot(kind="box", ax=axes[1], vert=False)
axes[1].set_title("Salary distribution (after capping at 95th percentile)")
axes[1].set_xlabel("Salary (USD)")

plt.tight_layout()
plt.savefig("output/cleaning_salary_outlier.png", dpi=120)
plt.show()

print(f"Cap applied at: {cap_value:,.0f} USD")

**When to cap vs when to drop:** Cap (winsorise) when the outlier is a data entry error but a valid record. Drop when the entire row is suspicious or meaningless.

---

## Section 9 - Step 7: Handle Missing Values

Now that types are fixed and outliers are handled, we deal with missing values. The right strategy depends on the column.

| Strategy | When to use |
|---|---|
| Drop the row | Missing value is in a critical column with no substitute |
| Fill with mean/median | Numeric column, missing at random, no strong group effect |
| Fill with group median | Numeric column where the value varies by a category |
| Fill with mode | Categorical column |
| Forward/backward fill | Time series data |
| Flag as "Unknown" | Categorical where missingness itself is informative |

In [ ]:
print("Missing values before imputation:")
print(df.isnull().sum().to_string())

In [ ]:
# Strategy 1: Drop rows where name is missing (cannot have an employee without a name)
df = df.dropna(subset=["name"]).reset_index(drop=True)
print(f"Rows after dropping missing names: {len(df)}")

In [ ]:
# Strategy 2: Fill department with mode (most common department)
dept_mode = df["department"].mode()[0]
df["department"] = df["department"].fillna(dept_mode)
print(f"Department NaN filled with mode: '{dept_mode}'")

In [ ]:
# Strategy 3: Fill salary using median per department
# Salaries differ by department, so a global median would be misleading
df["salary"] = df.groupby("department")["salary"].transform(
    lambda x: x.fillna(x.median())
)
print("Salary imputed with department-level median.")
print(df.groupby("department")["salary"].median().round(0).to_string())

In [ ]:
# Strategy 4: Fill age with global median
age_median = df["age"].median()
df["age"] = df["age"].fillna(age_median)
print(f"Age NaN filled with global median: {age_median}")

# Strategy 5: Fill performance_score with global median
perf_median = df["performance_score"].median()
df["performance_score"] = df["performance_score"].fillna(perf_median)
print(f"Performance score NaN filled with median: {perf_median}")

# Strategy 6: Join date - fill with NaT and flag it
df["join_date_missing"] = df["join_date"].isnull().astype(int)
print(f"\nRows with missing join_date flagged: {df['join_date_missing'].sum()}")

In [ ]:
print("Missing values after all imputation steps:")
print(df.isnull().sum().to_string())

---

## Section 10 - Step 8: Final Validation and Summary

In [ ]:
def cleaning_report(df_before, df_after):
    print("=" * 50)
    print("  CLEANING SUMMARY")
    print("=" * 50)
    print(f"  Rows before  : {len(df_before)}")
    print(f"  Rows after   : {len(df_after)}")
    print(f"  Rows removed : {len(df_before) - len(df_after)}")
    print(f"  Columns added: {len(df_after.columns) - len(df_before.columns)}")
    print(f"  Remaining NaN: {df_after.isnull().sum().sum()}")
    print("=" * 50)

cleaning_report(df_dirty, df)

In [ ]:
df.to_csv("output/employees_cleaned.csv", index=False)
print("Saved employees_cleaned.csv")
df.head(10)

---

## Section 11 - Building a Reusable Cleaning Pipeline

In production, cleaning steps should be wrapped in functions and applied in a defined order. This makes the pipeline reproducible and easy to update.

In [ ]:
def clean_employee_data(df_input):
    """
    Apply the full cleaning pipeline to a raw employee DataFrame.
    Returns a cleaned copy without modifying the input.
    """
    df = df_input.copy()

    # 1. Fix types
    df["salary"]    = pd.to_numeric(df["salary"], errors="coerce")
    df["join_date"] = pd.to_datetime(df["join_date"], errors="coerce")

    # 2. Fix strings
    df["name"]       = df["name"].str.strip().str.title()
    df["department"] = df["department"].str.strip().str.title()
    df["email"]      = df["email"].str.lower().str.strip()

    # 3. Remove duplicates
    df = df.drop_duplicates(subset="employee_id", keep="first")

    # 4. Validate business rules
    today = pd.Timestamp("today").normalize()
    df.loc[(df["age"] < 18) | (df["age"] > 80), "age"] = np.nan
    df.loc[df["join_date"] > today, "join_date"] = pd.NaT

    # 5. Cap outliers
    cap = df["salary"].quantile(0.95)
    df["salary"] = df["salary"].clip(upper=cap)

    # 6. Handle missing values
    df = df.dropna(subset=["name"])
    df["department"]         = df["department"].fillna(df["department"].mode()[0])
    df["salary"]             = df.groupby("department")["salary"].transform(lambda x: x.fillna(x.median()))
    df["age"]                = df["age"].fillna(df["age"].median())
    df["performance_score"]  = df["performance_score"].fillna(df["performance_score"].median())
    df["join_date_missing"]  = df["join_date"].isnull().astype(int)

    return df.reset_index(drop=True)


# Apply the pipeline
df_clean = clean_employee_data(df_dirty)
print(f"Pipeline output shape: {df_clean.shape}")
print(f"Remaining nulls: {df_clean.isnull().sum().sum()}")

---

## Section 12 - Summary of Cleaning Steps

| Step | Method used | Why |
|---|---|---|
| Fix data types | `pd.to_numeric(errors='coerce')`, `pd.to_datetime` | Enables numeric and date operations |
| Fix strings | `.str.strip()`, `.str.title()` | Collapses variant spellings into one category |
| Remove duplicates | `drop_duplicates(subset=key_col)` | Prevents inflated counts and averages |
| Validate rules | Conditional `np.nan` assignment | Removes logically impossible values |
| Cap outliers | `.clip(upper=quantile)` | Reduces distortion without deleting rows |
| Impute missing | Group median, mode, drop | Appropriate strategy per column type |

---

**Practice challenge:** Add a new column `years_of_service` computed from `join_date` to the dataset, then check if it correlates with `salary`. Handle rows where `join_date` is null.